# Variational Autoencoders (VAE) con MNIST

Un **Variational Autoencoder (VAE)** extiende el autoencoder estándar añadiendo una estructura probabilística al espacio latente.

## ¿Qué cambia respecto al Autoencoder estándar?

| | Autoencoder | VAE |
|---|---|---|
| Encoder produce | un punto `z` | parámetros `μ, σ` de una distribución |
| Espacio latente | determinístico | estocástico, con prior `N(0, I)` |
| Generación | No controlada | Muestreo desde `N(0, I)` |
| Loss | MSE | MSE + KL divergence |

```
         Encoder                           Decoder
x ──► [fc1]─[fc2] ──► μ, log σ²           
                           │                  
                    z = μ + σ·ε  ──► [fc3]─[fc4] ──► x̂
                       (ε ~ N(0,I))  
```

## El truco de reparametrización

Para poder hacer backpropagation a través del muestreo, usamos:

$$z = \mu + \sigma \odot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I)$$

El gradiente fluye a través de `μ` y `σ`, pero **no** a través de `ε`.

## La función de pérdida: ELBO

El VAE maximiza la **Evidence Lower Bound (ELBO)**:

$$\mathcal{L} = \underbrace{\mathbb{E}_{q(z|x)}[\log p(x|z)]}_{\text{Reconstrucción}} - \underbrace{D_{KL}(q(z|x) \| p(z))}_{\text{Regularización}}$$

- **Término de reconstrucción**: el decoder debe recuperar `x` a partir de `z`. Medido con MSE (o BCE).
- **Término KL**: fuerza a la distribución posterior `q(z|x) = N(μ, σ²)` a parecerse al prior `p(z) = N(0, I)`.

Para gaussianas diagonales, el KL tiene forma cerrada:

$$D_{KL} = -\frac{1}{2} \sum_{j=1}^{d} \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

## En este notebook:
1. Implementamos el VAE con el truco de reparametrización.
2. Entrenamos y monitoreamos **ambos términos** de la loss por separado.
3. Exploramos el **balance reconstrucción/KL** con β-VAE.
4. Visualizamos el **espacio latente** y la **generación** muestreando desde `N(0, I)`.
5. Comparamos con el autoencoder estándar.

## 0. Imports y configuración

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

# Dispositivo
if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
else:
    DEVICE = torch.device("cpu")

print(f"Usando dispositivo: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Dataset MNIST

In [ ]:
DATA_DIR = "../02_cnns/data"

transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(root=DATA_DIR, train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=transform)

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} | Test: {len(test_dataset)}")

## 2. Arquitectura del VAE

El encoder produce **dos vectores**: `μ` (media) y `log σ²` (log-varianza). El muestreo de `z` se hace con el truco de reparametrización para mantener el grafo diferenciable.

In [ ]:
class VAE(nn.Module):
    """Variational Autoencoder fully-connected para imágenes 28x28."""

    def __init__(self, latent_dim: int):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder compartido: 784 → 256 → 128
        self.encoder_shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
        )
        # Cabezas separadas para μ y log σ²
        self.fc_mu     = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # Decoder: latent_dim → 128 → 256 → 784
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid(),
        )

    def encode(self, x):
        """Devuelve (mu, log_var)."""
        h = self.encoder_shared(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, log_var):
        """z = mu + sigma * epsilon,  epsilon ~ N(0, I)."""
        if self.training:
            std = torch.exp(0.5 * log_var)   # σ = exp(log σ² / 2)
            eps = torch.randn_like(std)       # ε ~ N(0, I)
            return mu + std * eps
        else:
            return mu  # en inferencia usamos la media directamente

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        x_hat = self.decode(z)
        return x_hat, mu, log_var


# Verificación rápida
_vae = VAE(latent_dim=8).to(DEVICE)
_x = torch.randn(4, 1, 28, 28).to(DEVICE)
_xhat, _mu, _lv = _vae(_x)
print(f"Input: {_x.shape}  →  μ: {_mu.shape}  log_var: {_lv.shape}  →  Recon: {_xhat.shape}")
del _vae, _x, _xhat, _mu, _lv

## 3. Función de pérdida: ELBO

La loss total es:

$$\mathcal{L}_{\text{VAE}} = \underbrace{\text{MSE}(x, \hat{x})}_{\text{Reconstrucción}} + \beta \cdot \underbrace{D_{KL}(q(z|x)\,\|\,p(z))}_{\text{Regularización KL}}$$

donde $\beta = 1$ es el VAE estándar, y $\beta > 1$ es el **β-VAE** que enfatiza la regularización.

El KL cerrado para gaussianas diagonales:

$$D_{KL} = -\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

In [ ]:
def vae_loss(x_hat, x, mu, log_var, beta=1.0):
    """
    ELBO loss del VAE.
    Devuelve (loss_total, recon_loss, kl_loss) — todos normalizados por nro. de muestras.
    """
    N = x.size(0)

    # Término de reconstrucción (MSE por píxel, promediado sobre el batch)
    recon = nn.functional.mse_loss(x_hat, x, reduction="sum") / N

    # Término KL en forma cerrada, sumado sobre dimensiones latentes y promediado sobre el batch
    kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / N

    return recon + beta * kl, recon, kl

## 4. Entrenamiento

Entrenamos un VAE con `d=8` y monitoreamos por separado los dos términos de la loss para ver cómo evolucionan durante el entrenamiento.

In [ ]:
def train_epoch_vae(model, loader, optimizer, beta, device):
    model.train()
    total_loss = total_recon = total_kl = 0.0
    for x, _ in loader:
        x = x.to(device)
        x_hat, mu, log_var = model(x)
        loss, recon, kl = vae_loss(x_hat, x, mu, log_var, beta=beta)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        n = x.size(0)
        total_loss  += loss.item()  * n
        total_recon += recon.item() * n
        total_kl    += kl.item()    * n
    N = len(loader.dataset)
    return total_loss / N, total_recon / N, total_kl / N


@torch.no_grad()
def eval_vae(model, loader, beta, device):
    model.eval()
    total_loss = total_recon = total_kl = 0.0
    for x, _ in loader:
        x = x.to(device)
        x_hat, mu, log_var = model(x)
        loss, recon, kl = vae_loss(x_hat, x, mu, log_var, beta=beta)
        n = x.size(0)
        total_loss  += loss.item()  * n
        total_recon += recon.item() * n
        total_kl    += kl.item()    * n
    N = len(loader.dataset)
    return total_loss / N, total_recon / N, total_kl / N

In [ ]:
LATENT_DIM  = 8
NUM_EPOCHS  = 30
LR          = 1e-3
BETA        = 1.0   # VAE estándar

vae = VAE(latent_dim=LATENT_DIM).to(DEVICE)
optimizer = optim.Adam(vae.parameters(), lr=LR)

history = {"loss": [], "recon": [], "kl": [],
           "val_loss": [], "val_recon": [], "val_kl": []}

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_recon, tr_kl = train_epoch_vae(vae, train_loader, optimizer, BETA, DEVICE)
    vl_loss, vl_recon, vl_kl = eval_vae(vae, test_loader, BETA, DEVICE)

    history["loss"].append(tr_loss);   history["val_loss"].append(vl_loss)
    history["recon"].append(tr_recon); history["val_recon"].append(vl_recon)
    history["kl"].append(tr_kl);       history["val_kl"].append(vl_kl)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Época {epoch:3d}/{NUM_EPOCHS}  "
              f"Loss: {tr_loss:.4f}  Recon: {tr_recon:.4f}  KL: {tr_kl:.4f}  "
              f"| Val Loss: {vl_loss:.4f}")

print("\n✓ Entrenamiento completo.")

## 5. Balance reconstrucción / KL durante el entrenamiento

Graficamos por separado cómo evolucionan los dos términos de la loss:

- **Reconstrucción**: baja rápido al principio mientras el modelo aprende a copiar la entrada.
- **KL**: sube al principio (el encoder aprende a usar el espacio latente) y luego se estabiliza.

Al inicio el KL es casi 0 porque el encoder devuelve `μ≈0, σ≈1` (lo que coincide con el prior). A medida que aprende representaciones útiles, `μ` y `σ` se alejan del prior, aumentando el KL.

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss total
ax = axes[0]
ax.plot(epochs, history["loss"],     label="Train", linewidth=2)
ax.plot(epochs, history["val_loss"], label="Val",   linewidth=2, linestyle="--")
ax.set_title("Loss total (Recon + KL)")
ax.set_xlabel("Época"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(True, alpha=0.3)

# Reconstrucción
ax = axes[1]
ax.plot(epochs, history["recon"],     label="Train", color="steelblue", linewidth=2)
ax.plot(epochs, history["val_recon"], label="Val",   color="steelblue", linewidth=2, linestyle="--")
ax.set_title("Término de Reconstrucción (MSE)")
ax.set_xlabel("Época"); ax.set_ylabel("MSE")
ax.legend(); ax.grid(True, alpha=0.3)

# KL
ax = axes[2]
ax.plot(epochs, history["kl"],     label="Train", color="darkorange", linewidth=2)
ax.plot(epochs, history["val_kl"], label="Val",   color="darkorange", linewidth=2, linestyle="--")
ax.set_title("Término KL")
ax.set_xlabel("Época"); ax.set_ylabel("KL")
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle(f"Evolución de la loss durante el entrenamiento (β={BETA}, d={LATENT_DIM})",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. El balance β: reconstrucción vs. regularización

El hiperparámetro **β** controla el trade-off:

| β | Efecto |
|---|---|
| β → 0 | Ignora el KL → colapsa a autoencoder estándar |
| β = 1 | VAE estándar: balance entre reconstrucción y prior |
| β > 1 | **β-VAE**: énfasis en estructura latente interpretable, pero peor reconstrucción |

Entrenamos varios β y comparamos el trade-off final reconstrucción/KL.

In [ ]:
BETAS       = [0.1, 0.5, 1.0, 2.0, 5.0]
EPOCHS_BETA = 20   # entrenamiento más corto para la comparación

beta_results = {}   # beta → (final_recon, final_kl, model)

for beta in BETAS:
    model_b = VAE(latent_dim=LATENT_DIM).to(DEVICE)
    opt_b   = optim.Adam(model_b.parameters(), lr=LR)

    for epoch in range(1, EPOCHS_BETA + 1):
        train_epoch_vae(model_b, train_loader, opt_b, beta, DEVICE)

    _, val_recon, val_kl = eval_vae(model_b, test_loader, beta, DEVICE)
    beta_results[beta] = (val_recon, val_kl, model_b)
    print(f"  β={beta:.1f}  →  Recon: {val_recon:.5f}  KL: {val_kl:.4f}")

print("\n✓ Comparación de β completa.")

In [ ]:
recons = [beta_results[b][0] for b in BETAS]
kls    = [beta_results[b][1] for b in BETAS]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors_beta = cm.plasma(np.linspace(0.1, 0.9, len(BETAS)))

ax = axes[0]
bars = ax.bar([str(b) for b in BETAS], recons, color=colors_beta, edgecolor="black")
for bar, v in zip(bars, recons):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001,
            f"{v:.4f}", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("β"); ax.set_ylabel("MSE (test)")
ax.set_title("Error de reconstrucción")
ax.grid(True, axis="y", alpha=0.3)

ax = axes[1]
bars = ax.bar([str(b) for b in BETAS], kls, color=colors_beta, edgecolor="black")
for bar, v in zip(bars, kls):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{v:.2f}", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("β"); ax.set_ylabel("KL (test)")
ax.set_title("Término KL")
ax.grid(True, axis="y", alpha=0.3)

plt.suptitle("Trade-off reconstrucción / KL según β", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Curva de Pareto
fig, ax = plt.subplots(figsize=(6, 5))
for i, (b, color) in enumerate(zip(BETAS, colors_beta)):
    ax.scatter(kls[i], recons[i], color=color, s=120, zorder=5, label=f"β={b}")
ax.set_xlabel("KL divergence")
ax.set_ylabel("MSE (reconstrucción)")
ax.set_title("Curva de Pareto: Reconstrucción vs. KL")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Reconstrucciones

Comparamos originales vs. reconstrucciones del VAE (usando la media `μ` en inferencia).

In [ ]:
N_SHOW = 10

originals, _ = next(iter(test_loader))
originals = originals[:N_SHOW].to(DEVICE)

vae.eval()
with torch.no_grad():
    recons, _, _ = vae(originals)

fig, axes = plt.subplots(2, N_SHOW, figsize=(N_SHOW * 1.3, 3))

for j in range(N_SHOW):
    axes[0, j].imshow(originals[j].cpu().squeeze(), cmap="gray", vmin=0, vmax=1)
    axes[0, j].axis("off")
    axes[1, j].imshow(recons[j].cpu().squeeze(),    cmap="gray", vmin=0, vmax=1)
    axes[1, j].axis("off")

axes[0, 0].set_ylabel("Original",    fontsize=9, rotation=0, labelpad=45, va="center")
axes[1, 0].set_ylabel("Reconstrucción", fontsize=9, rotation=0, labelpad=60, va="center")

plt.suptitle(f"Reconstrucciones del VAE (d={LATENT_DIM}, β={BETA})",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Generación: muestreo desde el prior

La ventaja clave del VAE sobre el AE estándar: podemos **generar imágenes nuevas** muestreando `z ~ N(0, I)` y pasándolo por el decoder. Esto funciona porque el KL forzó al espacio latente a parecerse al prior.

In [ ]:
N_GEN = 64   # imágenes a generar

vae.eval()
with torch.no_grad():
    z_sample = torch.randn(N_GEN, LATENT_DIM).to(DEVICE)   # z ~ N(0, I)
    gen_imgs = vae.decode(z_sample).cpu()

grid = make_grid(gen_imgs, nrow=8, padding=2)
plt.figure(figsize=(12, 8))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray", vmin=0, vmax=1)
plt.axis("off")
plt.title(f"Imágenes generadas por muestreo z ~ N(0, I)  (d={LATENT_DIM}, β={BETA})",
          fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### 8.1 Efecto de β en la generación

Con β alto, el espacio latente está más cerca del prior → mejor generación. Con β bajo, el decoder es más fiel pero el espacio latente puede tener "huecos" que producen imágenes incoherentes al muestrear.

In [ ]:
torch.manual_seed(7)   # misma semilla para todos para comparar
z_fixed = torch.randn(16, LATENT_DIM).to(DEVICE)

n_betas_show = len(BETAS)
fig, axes = plt.subplots(n_betas_show, 16, figsize=(16 * 0.8, n_betas_show * 0.9))

for i, beta in enumerate(BETAS):
    model_b = beta_results[beta][2]
    model_b.eval()
    with torch.no_grad():
        imgs = model_b.decode(z_fixed).cpu()
    for j in range(16):
        axes[i, j].imshow(imgs[j].squeeze(), cmap="gray", vmin=0, vmax=1)
        axes[i, j].axis("off")
    axes[i, 0].set_ylabel(f"β={beta}", fontsize=9, rotation=0, labelpad=30, va="center")

plt.suptitle("Generación con mismo z para distintos β", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Visualización del espacio latente

Como el VAE fuerza `q(z|x) ≈ N(0, I)`, esperamos que las distribuciones de las distintas clases sean más **compactas y centradas** que en el AE estándar. Usamos t-SNE para visualizar.

In [ ]:
CMAP_10 = plt.cm.get_cmap("tab10", 10)

@torch.no_grad()
def get_vae_latent(model, loader, device, max_samples=5000):
    model.eval()
    mus, labs = [], []
    total = 0
    for x, y in loader:
        x = x.to(device)
        mu, _ = model.encode(x)
        mus.append(mu.cpu().numpy())
        labs.append(y.numpy())
        total += x.size(0)
        if total >= max_samples:
            break
    return np.concatenate(mus)[:max_samples], np.concatenate(labs)[:max_samples]


N_VIZ = 5000
mu_codes, mu_labels = get_vae_latent(vae, test_loader, DEVICE, max_samples=N_VIZ)

# t-SNE sobre los códigos μ
print("Calculando t-SNE...", end=" ", flush=True)
tsne = TSNE(n_components=2, perplexity=30, max_iter=500, random_state=42, n_jobs=-1)
z2d = tsne.fit_transform(mu_codes)
print("listo.")

fig, ax = plt.subplots(figsize=(8, 7))
for digit in range(10):
    mask = mu_labels == digit
    ax.scatter(z2d[mask, 0], z2d[mask, 1],
               s=5, alpha=0.5, color=CMAP_10(digit), label=str(digit))
ax.legend(title="Dígito", markerscale=3, fontsize=8, ncol=2)
ax.set_title(f"Espacio latente del VAE (t-SNE, d={LATENT_DIM})", fontsize=12)
ax.set_xlabel("z₁"); ax.set_ylabel("z₂")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### 9.1 Visualización directa con d=2

Con `d=2` podemos graficar directamente los valores de `μ`. Idealmente veremos clusters separados pero sin grandes huecos vacíos (gracias al prior gaussiano).

In [ ]:
# Entrenamos un VAE con d=2 para visualización directa
vae2 = VAE(latent_dim=2).to(DEVICE)
opt2 = optim.Adam(vae2.parameters(), lr=LR)

print("Entrenando VAE con d=2...")
for epoch in range(1, NUM_EPOCHS + 1):
    train_epoch_vae(vae2, train_loader, opt2, beta=1.0, device=DEVICE)
    if epoch % 10 == 0:
        _, vl_recon, vl_kl = eval_vae(vae2, test_loader, 1.0, DEVICE)
        print(f"  Época {epoch}/{NUM_EPOCHS}  Recon: {vl_recon:.5f}  KL: {vl_kl:.4f}")

print("✓ Listo.")

In [ ]:
mu2_codes, mu2_labels = get_vae_latent(vae2, test_loader, DEVICE, max_samples=N_VIZ)

fig, ax = plt.subplots(figsize=(8, 7))
for digit in range(10):
    mask = mu2_labels == digit
    ax.scatter(mu2_codes[mask, 0], mu2_codes[mask, 1],
               s=5, alpha=0.5, color=CMAP_10(digit), label=str(digit))
ax.legend(title="Dígito", markerscale=3, fontsize=8, ncol=2)
ax.set_title("Espacio latente del VAE (d=2) — proyección directa de μ", fontsize=12)
ax.set_xlabel("μ₁"); ax.set_ylabel("μ₂")
ax.grid(True, alpha=0.2)

# Contorno del prior N(0,I): elipses al 1σ, 2σ, 3σ
theta = np.linspace(0, 2 * np.pi, 200)
for r, ls in [(1, "--"), (2, ":"), (3, "-.")]:
    ax.plot(r * np.cos(theta), r * np.sin(theta), color="gray", linestyle=ls,
            linewidth=1, alpha=0.6, label=f"{r}σ prior")
ax.legend(title="Dígito / Prior", markerscale=3, fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

### 9.2 Mapa de generación 2D

Con `d=2` podemos barrer el espacio latente en una grilla y ver qué genera el decoder en cada punto.

In [ ]:
from scipy.stats import norm

N_GRID = 20   # resolución de la grilla

# Barremos cuantiles del prior para cubrir la zona de mayor probabilidad
grid_x = norm.ppf(np.linspace(0.05, 0.95, N_GRID))
grid_y = norm.ppf(np.linspace(0.05, 0.95, N_GRID))

canvas = np.zeros((N_GRID * 28, N_GRID * 28))

vae2.eval()
with torch.no_grad():
    for i, yi in enumerate(grid_y[::-1]):   # eje y de arriba a abajo
        for j, xj in enumerate(grid_x):
            z = torch.tensor([[xj, yi]], dtype=torch.float32).to(DEVICE)
            img = vae2.decode(z).cpu().squeeze().numpy()
            canvas[i * 28:(i + 1) * 28, j * 28:(j + 1) * 28] = img

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(canvas, cmap="gray", origin="upper",
          extent=[grid_x[0], grid_x[-1], grid_y[0], grid_y[-1]])
ax.set_title("Mapa de generación del VAE (d=2): decodificando grilla en el espacio latente",
             fontsize=12, fontweight="bold")
ax.set_xlabel("z₁"); ax.set_ylabel("z₂")
plt.tight_layout()
plt.show()

## 10. Interpolación en el espacio latente

Al igual que en el AE, podemos interpolar entre dos códigos. La diferencia es que en el VAE el espacio es más suave (gracias al prior), por lo que las transiciones son más coherentes.

In [ ]:
test_images_all = test_dataset.data.float() / 255.0
test_labels_all = test_dataset.targets

PAIRS   = [(0, 1), (3, 8), (4, 9), (2, 7)]
N_STEPS = 12

vae.eval()
fig, axes = plt.subplots(len(PAIRS), 1, figsize=(N_STEPS * 1.3, len(PAIRS) * 1.4))

for ax, (da, db) in zip(axes, PAIRS):
    ia = (test_labels_all == da).nonzero(as_tuple=True)[0][0]
    ib = (test_labels_all == db).nonzero(as_tuple=True)[0][0]
    img_a = test_images_all[ia].unsqueeze(0).unsqueeze(0).to(DEVICE)  # (1,1,28,28)
    img_b = test_images_all[ib].unsqueeze(0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        mu_a, _ = vae.encode(img_a)
        mu_b, _ = vae.encode(img_b)
        alphas  = torch.linspace(0, 1, N_STEPS).to(DEVICE)
        z_interp = torch.stack([(1 - a) * mu_a + a * mu_b for a in alphas]).squeeze(1)
        interp_imgs = vae.decode(z_interp).cpu()

    orig_a = test_images_all[ia].unsqueeze(0).unsqueeze(0)  # (1,1,28,28)
    orig_b = test_images_all[ib].unsqueeze(0).unsqueeze(0)
    strip = torch.cat([orig_a] + [interp_imgs[i].unsqueeze(0) for i in range(N_STEPS)] + [orig_b], dim=0)
    grid = make_grid(strip, nrow=len(strip), padding=2)
    ax.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
    ax.set_title(f"{da} → {db}", fontsize=9, loc="left")

plt.suptitle(f"Interpolación lineal en el espacio latente del VAE (d={LATENT_DIM})",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 11. Resumen y conclusiones

### ¿Qué aprendimos?

| Concepto | Observación |
|---|---|
| **Truco de reparametrización** | Permite backprop a través del muestreo estocástico. |
| **Término KL** | Empieza en 0, sube mientras el encoder aprende representaciones útiles, y se estabiliza. |
| **Término de reconstrucción** | Baja rápidamente; el modelo aprende primero a copiar la entrada. |
| **β chico** | Mejor reconstrucción, peor prior → huecos en el espacio latente. |
| **β grande** | Peor reconstrucción, mejor prior → generación más coherente. |
| **Generación** | Muestrear `z ~ N(0,I)` produce imágenes realistas (no era posible con el AE estándar). |
| **Mapa 2D** | El decoder aprende una función continua: zonas del espacio latente corresponden a dígitos. |

### Limitaciones del VAE

- Las imágenes generadas son **borrosas** (el MSE tiende a promediar modos del dato).
- El prior isotrópico `N(0, I)` puede ser demasiado simple para datos complejos.

### ¿Qué sigue?

- **Conditional VAE (CVAE)**: condicionar en la clase → generación controlada.
- **VQ-VAE**: espacio latente discreto → codecs de audio e imagen de alta calidad.
- **Generative Adversarial Networks (GANs)**: usar un discriminador en lugar del KL → imágenes más nítidas.
- **Latent Diffusion Models**: combina espacio latente del VAE con modelos de difusión → Stable Diffusion.